In [ ]:
import torch
import numpy as np
from tqdm.auto import tqdm

import mediapy as media

import matplotlib.pyplot as plt
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from minigridutils import *
from torch.utils.data import DataLoader, Dataset

In [ ]:
## GFlowNet Training

n_iterations = 8_000
batch_size = 32
alternate_every = 200
state_size = 3
dict_size = 5
net = EMGFlowNet(dict_size=dict_size, state_size=state_size)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for iter in range(n_iterations):
    
    observations, actions, all_actions, lengths, text_missions = generate_conditioning_sampless(batch_size)
    actions = torch.nn.utils.rnn.pad_sequence(actions, batch_first=True, padding_value=5)
    
    if iter == 1_000:
        net.gfn_optimizer.param_groups[0]["lr"] = 2e-5
    
    if (iter % (600)) < 300:
        gfn_loss = net.train_gfn(
            observations.to(device),
            actions.to(device),
            0.2
        )
    
    else:
        dec_loss = net.train_decoder(
            observations.to(device),
            actions.to(device)
        )
    
    if (iter+1) % 100 == 0:
        torch.save(net, "gfn")

In [ ]:
## VQ-VAE Training

n_iterations = 5_000
batch_size = 32
state_size = 5
dict_size = 8
embedding_size = 1
vae = SequentialVQVAE(state_size, dict_size, embedding_size)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vae.to(device)
optimizer = torch.optim.Adam(vae.parameters(), lr=0.0005)

for iter in range(n_iterations):

    observations, actions, all_actions, lengths, text_missions = generate_conditioning_sampless(batch_size)
    dec_loss, e_loss = vae(observations.to(device), torch.nn.utils.rnn.pad_sequence(actions, batch_first=True, padding_value=5).to(device))
    loss = dec_loss + e_loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (iter+1) % 100 == 0:
        torch.save(net, "vqvae")

In [ ]:
## VAE Training

n_iterations = 5_000
batch_size = 32
latent_dim = 5
vae = SequentialVAE(latent_dim=latent_dim)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vae.to(device)
optimizer = torch.optim.Adam(vae.parameters(), lr=0.001)

for iter in range(n_iterations):

    observations, actions, all_actions, lengths, text_missions = generate_conditioning_sampless(batch_size)
    dec_loss, kl_loss = vae(observations.to(device), torch.nn.utils.rnn.pad_sequence(actions, batch_first=True, padding_value=5).to(device))
    loss = dec_loss + 0.05 * kl_loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (iter+1) % 100 == 0:
        torch.save(net, "vae")

In [ ]:
# Testing GFlowNet

net = torch.load("gfn", weights_only=False, map_location=torch.device("cpu"))
net.device=torch.device("cpu")
observations, actions, all_actions, lengths, text_missions, frames = generate_conditioning_sampless(100, with_frames=True)
actions = torch.nn.utils.rnn.pad_sequence(actions, batch_first=True, padding_value=5)
labels = text_missions[:, 0].copy()
labels[text_missions[:, 0]=="green"] = 0
labels[text_missions[:, 0]=="blue"] = 1
labels[text_missions[:, 0]=="purple"] = 2
labels[text_missions[:, 0]=="yellow"] = 3
labels[text_missions[:, 0] == "switch"] = text_missions[text_missions[:, 0] == "switch", 1]
labels[np.logical_or(labels == "green", labels == "blue")] = 4
labels[np.logical_or(labels == "yellow", labels == "purple")] = 5

# We can take the labels as the overall agent type or the currently followed object
# labels = text_missions[:, 1:].copy()
# labels[text_missions[:, 1:]=="green"] = 0
# labels[text_missions[:, 1:]=="blue"] = 1
# labels[text_missions[:, 1:]=="purple"] = 2
# labels[text_missions[:, 1:]=="yellow"] = 3
labels = torch.from_numpy(labels.astype(int)).long()

observations = observations[:, :-1].to(net.device)
batch, seq, *channels = observations.shape
obs_features = net.convnet(
    net.emb(observations.reshape(batch * seq, *channels))
).reshape(batch, seq, -1)
dec_features = net.dec_convnet(
    net.dec_emb(observations.reshape(batch * seq, *channels))
).reshape(batch, seq, -1)
enc_h = torch.zeros(batch, 64, device=observations.device)
gfn_loss = torch.zeros(batch, seq - 1)
latents, preds, rewards = [], [], []
enc = []

with torch.no_grad():
    for t in range(seq - 1):
        enc_h = net.encoder_lstm(obs_features[:, t], enc_h)
        enc.append(enc_h)
        forward_terminal_states, forward_log_pf = (
            net.sample_ar_trajectories(conditioning=enc_h, rand_prob=0, prob_exponent=-1)
        )
        # forward_terminal_states = torch.randint(0, net.dict_size, (batch, net.state_size))
        # forward_terminal_states = torch.zeros((batch, net.state_size)).long()
        # latents.append(forward_terminal_states)
        z = net.codebook[
            torch.arange(net.state_size, device=net.device)
            .unsqueeze(0)
            .expand(batch, net.state_size),
            forward_terminal_states,
        ].view(batch, -1)
        dec_latents = net.latent_mlp(z.to(net.device))
        latents.append(z)
        gamma, beta = dec_latents.chunk(2, dim=-1)
        gamma = 1 + gamma
        film_out = dec_features[:, t:] * gamma.unsqueeze(1).repeat(
            1, seq - t, 1
        ) + beta.unsqueeze(1).repeat(1, seq - t, 1)
        logprobs = net.decoder(torch.nn.functional.relu(film_out))[
            :, : seq - t - 1
        ]
        preds.append(logprobs)
        crewards = (
            -torch.nn.functional.cross_entropy(
                logprobs.flatten(0, 1),
                actions[:, t:].flatten(0, 1),
                reduction="none",
            )
            .reshape(batch, seq - t - 1)
            .sum(-1)
        )
        rewards.append(crewards)
latents = torch.stack(latents, dim=1)

In [ ]:
class SLDataset(Dataset):
    def __init__(self, latents, labels):
        self.latents = latents
        self.labels = labels
    def __getitem__(self, index):
        return (self.latents[index], self.labels[index], index)
    def __len__(self):
        return len(self.latents)

dataset = SLDataset(
    latents.flatten(0, 1).float(),
    labels.repeat_interleave(seq-1, dim=0)
    # labels.flatten()
)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)
# pred_net = MLP(net.dict_size*net.state_size, 6, [128, 128])
pred_net = MLP(net.state_size, 6, [128, 128])
optimizer = torch.optim.Adam(pred_net.parameters(), lr=1e-3)
for epoch in (pbar:=tqdm(range(50))):
    losses = 0.0
    for latent, label, _ in dataloader:
        out = pred_net(latent)
        loss = torch.nn.functional.cross_entropy(out, label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses += loss.item()
    pbar.set_description(f"Loss: {losses:.4f}")

In [ ]:
plt.plot(
    (
        pred_net(
            latents
        ).softmax(-1).argmax(-1) == labels.unsqueeze(-1).repeat_interleave(seq-1, dim=-1)
    ).sum(0) / batch
)

In [ ]:
step = 0
index = 18
media.show_video(frames[index], fps=2, loop=False)
preds[step][index].softmax(-1).argmax(-1), actions[index, step:]

In [ ]:
# Testing VAE / VQ-VAE

vae = SequentialVAE(5)
# vae = SequentialVQVAE(5, 8, 1)
vae.load_state_dict(torch.load("vae", map_location=torch.device("cpu")))
observations, actions, all_actions, lengths, text_missions, frames = generate_conditioning_sampless(100, with_frames=True)
actions = torch.nn.utils.rnn.pad_sequence(actions, batch_first=True, padding_value=5)
labels = text_missions[:, 0].copy()
labels[text_missions[:, 0]=="green"] = 0
labels[text_missions[:, 0]=="blue"] = 1
labels[text_missions[:, 0]=="purple"] = 2
labels[text_missions[:, 0]=="yellow"] = 3
labels[text_missions[:, 0] == "switch"] = text_missions[text_missions[:, 0] == "switch", 1]
labels[np.logical_or(labels == "green", labels == "blue")] = 4
labels[np.logical_or(labels == "yellow", labels == "purple")] = 5
# labels = text_missions[:, 1:].copy()
# labels[text_missions[:, 1:]=="green"] = 0
# labels[text_missions[:, 1:]=="blue"] = 1
# labels[text_missions[:, 1:]=="purple"] = 2
# labels[text_missions[:, 1:]=="yellow"] = 3
labels = torch.from_numpy(labels.astype(int)).long()

In [ ]:
obs = observations[:, :-1]
lengths = observations[:, -1, 0, 0, 0]
batch, seq, *channels = obs.shape
obs_features = vae.convnet(
    vae.emb(obs.reshape(batch * seq, *channels))
).reshape(batch, seq, -1)
enc_h = torch.zeros(batch, 64)
latents, outs, means, ind = [], [], [], []
for t in range(seq - 1):
    enc_h = vae.encoder_lstm(obs_features[:, t], enc_h)
    if vae._get_name() == "SequentialVAE":
        mu_q = vae.latent_mu(enc_h)
        logvar_q = vae.latent_logvar(enc_h)
        z_t = mu_q + torch.randn_like(logvar_q) * torch.exp(0.5 * logvar_q)
        means.append(mu_q)
    if vae._get_name() == "SequentialVQVAE":
        z = vae.latent_pre(enc_h).view(batch, vae.state_size, vae.embedding_size)
        indices = (
            (z.unsqueeze(2) - vae.codebook.unsqueeze(0).unsqueeze(0))
            .pow(2)
            .sum(-1)
            .argmin(-1)
        )
        z_t = vae.codebook[
            torch.arange(vae.state_size, device=z.device)
            .unsqueeze(0)
            .expand(batch, vae.state_size),
            indices,
        ].view(batch, -1)
        ind.append(indices)
    latents.append(z_t)
    gamma, beta = vae.latent_mlp(z_t).chunk(2, dim=-1)
    gamma = 1 + gamma
    film_out = obs_features[:, t:] * gamma.unsqueeze(1).repeat(
        1, seq - t, 1
    ) + beta.unsqueeze(1).repeat(1, seq - t, 1)

    out = vae.decoder(torch.nn.functional.relu(film_out))[:, : seq - t - 1]
    outs.append(out)
latents = torch.stack(latents, dim=1)
if vae._get_name() == "SequentialVAE":
    means = torch.stack(means, dim=1)
if vae._get_name() == "SequentialVQVAE":
    ind = torch.stack(ind, dim=1)

In [ ]:
class SLDataset(Dataset):
    def __init__(self, latents, labels):
        self.latents = latents
        self.labels = labels
    def __getitem__(self, index):
        return (self.latents[index], self.labels[index], index)
    def __len__(self):
        return len(self.latents)

dataset = SLDataset(latents.reshape(-1, latents.shape[-1]).detach(), labels.repeat_interleave(lengths, dim=0))
# dataset = SLDataset(ind.reshape(-1, vae.state_size) / vae.dict_size, labels.repeat_interleave(lengths, dim=0))
# dataset = SLDataset(ind.reshape(-1, vae.state_size) / vae.dict_size, labels.flatten())
# dataset = SLDataset(latents.reshape(-1, latents.shape[-1]).detach(), labels.flatten())
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)
in_dim = vae.state_size*vae.embedding_size if vae._get_name() == "SequentialVQVAE" else vae.latent_dim if vae._get_name() == "SequentialVAE" else None
net = MLP(in_dim, len(labels.unique()), [32, 32])
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
for epoch in (pbar:=tqdm(range(100))):
    losses = 0.0
    for latent, label, _ in dataloader:
        out = net(latent)
        loss = torch.nn.functional.cross_entropy(out, label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses += loss.item()
    pbar.set_description(f"Loss: {losses:.4f}")
plt.plot((net(latents).softmax(-1).argmax(-1) == labels.unsqueeze(-1).repeat_interleave(seq-1, dim=-1)).sum(0) / latents.shape[0])

In [ ]:
media.show_video(frames[18], fps=2, loop=False)

In [ ]:
step = 2
index = 10
outs[step][index].softmax(-1).argmax(-1), actions[index, step:]

In [ ]:
# step = 30
# plt.scatter(latents[:, step, 5].detach().numpy(), latents[:, step, 0].detach().numpy(), c=labels)
index = np.random.randint(50)
# index=4
plt.scatter(latents[index, :, 0].detach().numpy(), latents[index, :, 4].detach().numpy(), c=np.arange(50)) # ["blue", "red", "green", "grey"][labels[index]]
plt.xlim(-10, 10), plt.ylim(-10, 10)
plt.show()

In [ ]:
# Training a decoder to predict the policy without latents

class Decoder(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.dec_emb = ImageBOWEmbedding(147, 128).to(self.device)
        self.dec_convnet = torch.nn.Sequential(
            torch.nn.Conv2d(128, 128, kernel_size=(3, 3), stride=1, padding=1),
            torch.nn.BatchNorm2d(128),
            torch.nn.ReLU(),
            # torch.nn.Conv2d(128, 128, kernel_size=(3, 3), stride=1, padding=1),
            # torch.nn.BatchNorm2d(128),
            # torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=(7, 7), stride=2),
            torch.nn.ReLU(),
            torch.nn.Flatten(),
        ).to(self.device)
        self.dropout = torch.nn.Dropout()
        self.decoder = MLP(128, 6, hidden_sizes=[128, 128]).to(self.device)

    def forward(self, observations, actions):
        observations = observations[:, :-1].to(self.device)
        batch, seq, *channels = observations.shape
        dec_features = self.dec_convnet(
            self.dec_emb(observations.reshape(batch * seq, *channels))
        ).reshape(batch, seq, -1)
        logprobs = self.decoder(self.dropout(dec_features))[:, :-1, :]
        dec_loss = torch.nn.functional.cross_entropy(
            logprobs.flatten(0, 1),
            actions.flatten(0, 1),
            reduction="sum",
        )
        return dec_loss / batch

batch = 32
dec = Decoder()
decoder_optimizer = torch.optim.Adam(dec.parameters(), lr=0.0003)
losses = []
for epoch in (pbar:=tqdm(range(150))):
    observations, actions, all_actions, lengths, text_missions = generate_conditioning_sampless(batch)
    actions = torch.nn.utils.rnn.pad_sequence(actions, batch_first=True, padding_value=5)
    dec_loss = dec(observations, actions)
    decoder_optimizer.zero_grad()
    dec_loss.backward()
    decoder_optimizer.step()
    losses.append(dec_loss.item())
    pbar.set_description(f"Loss: {dec_loss.item():.4f}")

In [ ]:
observations, actions, all_actions, lengths, text_missions, frames = generate_conditioning_sampless(5, True)
observations = observations[:, :-1].to(dec.device)
actions = torch.nn.utils.rnn.pad_sequence(actions, batch_first=True, padding_value=5)
batch, seq, *channels = observations.shape
dec_features = dec.dec_convnet(
    dec.dec_emb(observations.reshape(batch * seq, *channels))
).reshape(batch, seq, -1)
logprobs = dec.decoder(dec.dropout(dec_features))[:, :-1, :]

In [ ]:
index = np.random.randint(5)
media.show_video(frames[index], fps=2, loop=False)
logprobs[index].softmax(-1).argmax(-1), actions[index]